In [ ]:
import asyncio
import struct

import ipywidgets
from IPython.display import display

import xyg
from test_geo_mounted_temporal import mounted_chart_fixture

source, chart = mounted_chart_fixture()
widget = chart.widget()
controller = xyg.TemporalController(instance_id=32, domain=(-(1 << 63), (1 << 63) - 1))
binding = widget.bind_temporal(controller)
controller.set_cursor(3)
status = ipywidgets.HTML()
display(widget, status)

In [ ]:
async def phase_1():
    global adapter, frame, caller, reader, good, binding, old, original_current
    await binding.when_idle()
    assert binding.error is None, binding.error
    original_current = source.current
    adapter = widget._adapter
    frame = adapter._frame
    assert frame.data.identity["time"]["instant"] == 3
    assert frame.data.record(0)["feature_id"] == (1 << 64) - 1
    caller = frame.retain()
    hits = frame.pick(style=frame._style, x=400, y=300, tolerance=0, mode=1, max_hits=10)
    assert any(
        struct.unpack_from("<Q", hits.data["records"], i * 48 + 8)[0] == (1 << 64) - 1
        for i in range(hits.data["count"])
    )
    hits.close()
    i = frame.data.identity
    await asyncio.wrap_future(
        widget.update(
            operation=3,
            args=[40, 0],
            sequence=i["sequence"] + 1,
            camera_revision=i["camera_revision"] + 1,
            time_revision=i["time_revision"],
            state_revision=i["state_revision"],
            time=i["time"],
        )
    )
    await asyncio.to_thread(controller.set_cursor, 4)
    await binding.when_idle()
    assert adapter._frame.data.identity["camera"]["center_x"] != 0
    assert adapter._frame.data.identity["time"]["instant"] == 4
    status.value = "XYG mounted time and camera accepted"


phase_task = asyncio.create_task(phase_1())
phase_task.add_done_callback(
    lambda task: setattr(status, "value", repr(task.exception())) if task.exception() else None
)

In [ ]:
async def phase_2():
    global adapter, frame, caller, reader, good, binding, old
    reader = source._reader
    source._reader = lambda ticket: (_ for _ in ()).throw(OSError("notebook injected read failure"))
    good = adapter._frame
    controller.set_cursor(5)
    await binding.when_idle()
    assert binding.error is not None
    assert adapter._frame is good
    source._reader = reader
    controller.set_cursor(6)
    await binding.when_idle()
    assert binding.error is None, binding.error
    good = adapter._frame

    def cancelled_reader(ticket):
        raise asyncio.CancelledError("notebook reader-originated cancellation")

    source._reader = cancelled_reader
    controller.set_cursor(7)
    await binding.when_idle()
    assert binding.error is not None
    assert adapter._frame is good
    source._reader = reader
    controller.set_cursor(8)
    await binding.when_idle()
    assert binding.error is None, binding.error
    await binding.aclose()
    controller.set_range(100000, 100001)
    binding = widget.bind_temporal(controller, "range")
    await binding.when_idle()
    assert binding.error is None, binding.error
    assert adapter._frame.data.identity["time"]["kind"] == 2
    status.value = "XYG empty range accepted"


phase_task = asyncio.create_task(phase_2())
phase_task.add_done_callback(
    lambda task: setattr(status, "value", repr(task.exception())) if task.exception() else None
)

In [ ]:
async def phase_3():
    global adapter, frame, caller, reader, good, binding, old
    await binding.aclose()
    controller.set_cursor(6)
    binding = widget.bind_temporal(controller)
    await binding.when_idle()
    assert binding.error is None, binding.error
    assert adapter._frame.data.record(0)["feature_id"] == (1 << 64) - 1
    assert caller.data.record(0)["feature_id"] == (1 << 64) - 1
    await binding.aclose()
    await binding.aclose()
    controller.close()
    caller.close()
    assert source.current is original_current
    status.value = "XYG rebind and independent caller accepted"


phase_task = asyncio.create_task(phase_3())
phase_task.add_done_callback(
    lambda task: setattr(status, "value", repr(task.exception())) if task.exception() else None
)

In [ ]:
async def phase_4():
    global adapter, frame, caller, reader, good, binding, old
    old = adapter._frame
    widget.close()
    while adapter.mounted:
        await asyncio.sleep(0.01)
    await widget._geo_rpc_task
    assert adapter._anchor is None
    try:
        _ = old.data
        raise AssertionError("frame remains after native release ACK")
    except RuntimeError:
        pass
    source.close()
    status.value = "XYG geographic browser release acknowledged"


phase_task = asyncio.create_task(phase_4())
phase_task.add_done_callback(
    lambda task: setattr(status, "value", repr(task.exception())) if task.exception() else None
)